In [ ]:
import os
from dotenv import load_dotenv
import fitz 
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAI
from langchain.chains.conversational_retrieval.base import ConversationalRetrievalChain
from langchain.docstore.document import Document
from langchain.memory import ConversationBufferMemory

In [ ]:
# Set your Google API key

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [ ]:
# Extract text from PDF
def extract_text_from_pdf(file_path):
    text = ""
    with fitz.open(file_path) as pdf:
        for page in pdf:
            text += page.get_text()
    return text

pdf_text = extract_text_from_pdf("data/Kishor_G.pdf")

In [ ]:
# Chunk text into LangChain Documents
chunks = pdf_text.split("\n\n")  

docs = [
    Document(page_content=chunk.strip())
    for chunk in chunks if chunk.strip()
]

In [ ]:
# Generate embeddings & store in FAISS
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
vectorstore.save_local("faiss_index")  

C:\Users\Admin\AppData\Local\Temp\ipykernel_5040\523300327.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load FAISS index
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
# Load Gemini LLM
llm = GoogleGenerativeAI(
    model="models/gemini-1.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

In [ ]:
# Setup conversation memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_5040\4111481334.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [ ]:
# Create QA Chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory
)

In [ ]:
# Ask a question
query = "What are the technical skills mentioned in the resume?"
result = qa_chain({"question": query})

print("Response:", result["answer"])

Response: The resume lists the following technical skills:

**Programming Languages:** JavaScript, Python, Java, C

**Frontend:** React.js, React Native, Expo, Tailwind CSS

**Backend:** Node.js, Express.js, FastAPI

**Database:** MongoDB, Firebase

**Version Control & DevOps:** Jira, Git, Github

**Machine Learning & Artificial Intelligence:** TensorFlow, Keras, scikit-learn, OpenCV, NLTK, Hugging Face Transformers, Natural language processing

**Data Science & Analytics:** Pandas, NumPy, Matplotlib, Seaborn
